In [1]:
import ast
from pathlib import Path

import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

BASE_PATH = Path('../data/results/complete_prompt')
GROUND_TRUTH_PATH = '../data/results/labeled_examples.csv'
OUTPUT_PATH = BASE_PATH / 'model_comparison_results.csv'

MODEL_SIZES = {'GPT-5.4': '', 'GPT-5.4-mini': '-mini', 'GPT-5.4-nano': '-nano'}
EFFORTS = ['medium', 'high']

MODEL_PATHS = {
    f'{model_name}-{effort}': (
        BASE_PATH / f'extracted_questions_context_api_{model_name.lower()}_{effort}.csv'
    )
    for model_name in MODEL_SIZES
    for effort in EFFORTS
}

# Modelo usado anteriormente, sem o sufixo de effort.
MODEL_PATHS['GPT-5.4-nano-no-reasoning'] = (
    BASE_PATH / 'extracted_questions_context_api_sample.csv'
)

In [2]:
def question_label(value):
    if pd.isna(value):
        raise ValueError('Valor ausente na coluna perguntas')

    questions = ast.literal_eval(value)
    return 'tem_pergunta' if questions else 'nao_pergunta'


def compare_model(df_model, df_labeled, model_name):
    df = df_labeled.merge(df_model, on='comment_id', how='inner', validate='one_to_one')

    if len(df) != len(df_labeled):
        raise ValueError(f'{model_name}: há comment_id do ground truth sem previsão')

    y_true = df['label_true']
    y_pred = df['label_pred']
    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=['nao_pergunta', 'tem_pergunta'],
    ).ravel()

    return {
        'model_name': model_name,
        'n_examples': len(df),
        'accuracy': accuracy_score(y_true, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'precision_tem_pergunta': precision_score(y_true, y_pred, pos_label='tem_pergunta', zero_division=0),
        'recall_tem_pergunta': recall_score(y_true, y_pred, pos_label='tem_pergunta', zero_division=0),
        'f1_tem_pergunta': f1_score(y_true, y_pred, pos_label='tem_pergunta', zero_division=0),
        'f1_macro': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'true_negative': tn,
        'false_positive': fp,
        'false_negative': fn,
        'true_positive': tp,
    }

In [3]:
df_labeled = pd.read_csv(GROUND_TRUTH_PATH, usecols=['text', 'label'])
df_labeled['comment_id'] = df_labeled['text'].str.extract(r'(?m)^ID:\s*(\S+)')
df_labeled = df_labeled.rename(columns={'label': 'label_true'})

valid_labels = ['nao_pergunta', 'tem_pergunta']
invalid_labels = ~df_labeled['label_true'].isin(valid_labels)
print(f'Exemplos ignorados por falta de rótulo válido: {invalid_labels.sum()}')

df_labeled = df_labeled.loc[~invalid_labels, ['comment_id', 'label_true']].copy()

if df_labeled['comment_id'].isna().any() or df_labeled['comment_id'].duplicated().any():
    raise ValueError('O ground truth possui comment_id ausente ou duplicado')

df_labeled['label_true'].value_counts()

Exemplos ignorados por falta de rótulo válido: 0


label_true
nao_pergunta    480
tem_pergunta     20
Name: count, dtype: int64

In [4]:
results = []

for model_name, path in MODEL_PATHS.items():
    if not path.exists():
        print(f'Arquivo não encontrado, modelo ignorado: {path.name}')
        continue

    df_model = pd.read_csv(path, usecols=['comment_id', 'perguntas'])

    if df_model['comment_id'].isna().any() or df_model['comment_id'].duplicated().any():
        raise ValueError(f'{model_name}: comment_id ausente ou duplicado')

    df_model['label_pred'] = df_model['perguntas'].apply(question_label)
    results.append(compare_model(df_model[['comment_id', 'label_pred']], df_labeled, model_name))

In [5]:
df_results = pd.DataFrame(results).sort_values('f1_tem_pergunta', ascending=False)
df_results.to_csv(OUTPUT_PATH, index=False)
df_results

,model_name,n_examples,accuracy,balanced_accuracy,precision_tem_pergunta,recall_tem_pergunta,f1_tem_pergunta,f1_macro,true_negative,false_positive,false_negative,true_positive
3,GPT-5.4-mini-high,500,0.978,0.892708,0.695652,0.80,0.744186,0.866346,473,7,4,16
0,GPT-5.4-medium,500,0.976,0.843750,0.700000,0.70,0.700000,0.843750,474,6,6,14
2,GPT-5.4-mini-medium,500,0.974,0.866667,0.652174,0.75,0.697674,0.842045,472,8,5,15
1,GPT-5.4-high,500,0.972,0.865625,0.625000,0.75,0.681818,0.833587,471,9,5,15
5,GPT-5.4-nano-high,500,0.936,0.894792,0.369565,0.85,0.515152,0.740445,451,29,3,17
4,GPT-5.4-nano-medium,500,0.928,0.914583,0.346154,0.90,0.500000,0.730603,446,34,2,18
6,GPT-5.4-nano-no-reasoning,500,0.892,0.919792,0.263889,0.95,0.413043,0.676786,427,53,1,19


## Avaliação do resultado final do pipeline em duas etapas

Esta avaliação usa a mesma amostra anotada. Comentários que não aparecem no arquivo final são tratados como `nao_pergunta`, pois foram descartados pelo classificador ou não geraram perguntas.

In [ ]:
TWO_STAGE_PATHS = {
    f'pipeline-duas-etapas-{effort}': Path(
        f'../data/results/extracted_question/relevant_question_extraction_gpt-5-4-mini_{effort}.csv'
    )
    for effort in ['medium', 'high']
}

two_stage_results = []

for model_name, path in TWO_STAGE_PATHS.items():
    if not path.exists():
        print(f'Arquivo não encontrado, modelo ignorado: {path}')
        continue

    df_two_stage = pd.read_csv(path, usecols=['comment_id', 'perguntas'])

    # O arquivo final contém apenas os comentários que chegaram à extração.

    if df_two_stage['comment_id'].isna().any() or df_two_stage['comment_id'].duplicated().any():
        raise ValueError(f'{model_name}: comment_id ausente ou duplicado')

    df_two_stage['label_pred'] = df_two_stage['perguntas'].apply(question_label)

    # Mantém todos os comentários anotados; os ausentes no resultado são negativos.
    df_two_stage_eval = df_labeled[['comment_id', 'label_true']].merge(
        df_two_stage[['comment_id', 'label_pred']],
        on='comment_id',
        how='left',
    )
    df_two_stage_eval['label_pred'] = df_two_stage_eval['label_pred'].fillna('nao_pergunta')

    y_true = df_two_stage_eval['label_true']
    y_pred = df_two_stage_eval['label_pred']

    two_stage_results.append({
        'model_name': model_name,
        'n_examples': len(df_two_stage_eval),
        'accuracy': accuracy_score(y_true, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'precision_tem_pergunta': precision_score(y_true, y_pred, pos_label='tem_pergunta', zero_division=0),
        'recall_tem_pergunta': recall_score(y_true, y_pred, pos_label='tem_pergunta', zero_division=0),
        'f1_tem_pergunta': f1_score(y_true, y_pred, pos_label='tem_pergunta', zero_division=0),
        'f1_macro': f1_score(y_true, y_pred, average='macro', zero_division=0),
    })

df_two_stage_results = pd.DataFrame(two_stage_results).sort_values('f1_tem_pergunta', ascending=False)
df_two_stage_results

,model_name,n_examples,accuracy,balanced_accuracy,precision_tem_pergunta,recall_tem_pergunta,f1_tem_pergunta,f1_macro
1,pipeline-duas-etapas-high,500,0.978,0.892708,0.695652,0.80,0.744186,0.866346
0,pipeline-duas-etapas-medium,500,0.974,0.866667,0.652174,0.75,0.697674,0.842045
